# Week 1: TTFT vs prompt length, cold vs warm

Sends each prompt twice: once uncached (**cold**), then right away again (**warm**). The gap shows
what the prefix cache saves, and the cold slope gives the GPU's **prefill rate**.

**Before running**
- This uses GPU time. Kaggle counts against your 30 hours per week, and Colab A100 spends compute units.
- Kaggle: Accelerator T4 x2, Internet on, and `HF_TOKEN` attached under Add-ons -> Secrets.
- Colab: A100, and `HF_TOKEN` added under the key icon.
- **Run the last cell when finished** so the GPU stops being used.

Try `RUN_CONFIG = "t4_qwen05b"` first. It's small and quick, so it proves the pipeline works before
you spend time on the bigger model.

In [ ]:
# Settings: this is the only cell you should need to edit.
import os
from datetime import UTC, datetime
from pathlib import Path

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
assert IS_KAGGLE or IS_COLAB, "Expected Kaggle or Colab"

# Choices: t4_qwen05b (smoke test), t4_llama32_3b (Week 1 on Kaggle), a100_llama31_8b (Colab A100)
RUN_CONFIG = "t4_llama32_3b" if IS_KAGGLE else "a100_llama31_8b"
GIT_REF = "week1-ttft-client"  # Change to main after merging
VLLM_VERSION = "0.33.0"
CROSSCHECK_PROMPT_TOKENS = 2000  # A length present in every sweep

# To resume a stopped run, paste its ID here.
RUN_ID = None
RUN_ID = RUN_ID or f"{datetime.now(UTC):%Y%m%dT%H%M%SZ}_{RUN_CONFIG}"

REPO_DIR = Path("/tmp/project-Anton")
print("platform:", "kaggle" if IS_KAGGLE else "colab")
print("config:  ", RUN_CONFIG)
print("run id:  ", RUN_ID)

In [ ]:
# Helpers, and a look at the GPU.
import subprocess


def sh(cmd, check=True, cwd=None):
    """Run a command and show its output as it runs."""
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=check, cwd=cwd)


def anton(*args, check=True):
    return sh(["uv", "run", "--frozen", "--no-dev", "anton", *args], check=check, cwd=REPO_DIR)


sh(["nvidia-smi"])

In [ ]:
# Load the Hugging Face token from platform secrets. Never print it or write it to a file.
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "HF_TOKEN is empty"
print("HF_TOKEN loaded")

In [ ]:
# Choose where results are saved. Colab uses Drive, so they survive a disconnect.
if IS_KAGGLE:
    RESULTS_DIR = Path("/kaggle/working/results")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    RESULTS_DIR = Path("/content/drive/MyDrive/project-Anton/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = RESULTS_DIR / RUN_ID
print("results:", RUN_DIR)

In [ ]:
# Install. The harness gets its own virtualenv, so its packages can't break vLLM's.
os.environ["UV_PYTHON"] = "3.12"
# One GPU only: a second T4 would change what's being measured.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if not REPO_DIR.exists():
    sh(["git", "clone", "--depth", "1", "--branch", GIT_REF,
        "https://github.com/TejasVaradaraj/project-Anton.git", REPO_DIR])
sh(["pip", "install", "-q", "uv==0.12.3", f"vllm[bench]=={VLLM_VERSION}"])
sh(["uv", "sync", "--frozen", "--no-dev"], cwd=REPO_DIR)
sh(["vllm", "--version"])
anton("--help")

In [ ]:
# Start vLLM and wait until it's ready. The first start downloads the model.
SERVER = f"configs/servers/{RUN_CONFIG}.yaml"
SWEEP = f"configs/sweeps/week1_{RUN_CONFIG}.yaml"
anton("server", "start", "--config", SERVER)

In [ ]:
# Run the sweep. If it stops, set RUN_ID in the first cell and run this again to continue.
anton("ttft", "run", "--server", SERVER, "--sweep", SWEEP,
      "--run-id", RUN_ID, "--results-dir", RESULTS_DIR)

In [ ]:
# Analyze: writes summary.csv, analysis.json and the plot.
import pandas as pd
from IPython.display import Image, display

anton("ttft", "analyze", RUN_DIR)
display(pd.read_csv(RUN_DIR / "summary.csv"))
display(Image(RUN_DIR / "ttft_vs_prompt.png"))

## Cross-check against `vllm bench serve`

A second tool, written by other people, measuring the same thing. The two medians should be close.
A large gap means something is wrong with one of them, so find out why before trusting the numbers.
Don't expect an exact match, since the prompts differ.

In [ ]:
# Cross-check against vLLM's own benchmark, one request at a time, with prompts of the same length.
import json

import yaml

cfg = yaml.safe_load((REPO_DIR / SERVER).read_text())
sweep = yaml.safe_load((REPO_DIR / SWEEP).read_text())
crosscheck_dir = RUN_DIR / "raw"
result = sh([
    "vllm", "bench", "serve",
    "--backend", "openai", "--endpoint", "/v1/completions",
    "--host", "127.0.0.1", "--port", cfg["port"], "--model", cfg["model"],
    "--dataset-name", "random",
    "--random-input-len", CROSSCHECK_PROMPT_TOKENS,
    "--random-output-len", sweep["max_tokens"],
    "--num-prompts", 20, "--max-concurrency", 1, "--ignore-eos",
    "--save-result", "--result-dir", crosscheck_dir, "--result-filename", "vllm_bench.json",
], check=False)

if result.returncode == 0:
    bench = json.loads((crosscheck_dir / "vllm_bench.json").read_text())
    summary = pd.read_csv(RUN_DIR / "summary.csv")
    ours = summary[(summary.phase == "cold") & (summary.prompt_tokens == CROSSCHECK_PROMPT_TOKENS)]
    comparison = {
        "prompt_tokens": CROSSCHECK_PROMPT_TOKENS,
        "anton_cold_ttft_p50_ms": float(ours.ttft_p50_ms.iloc[0]),
        "vllm_bench_ttft_p50_ms": bench.get("median_ttft_ms"),
        "anton_tpot_p50_ms": float(ours.tpot_p50_ms.iloc[0]),
        "vllm_bench_tpot_p50_ms": bench.get("median_tpot_ms"),
    }
    (RUN_DIR / "crosscheck.json").write_text(json.dumps(comparison, indent=2))
    print(json.dumps(comparison, indent=2))
else:
    print("vllm bench serve failed, so there is NO cross-check for this run. See the output above.")

In [ ]:
# Save a copy with the raw data.
import shutil

archive = shutil.make_archive(str(RUN_DIR), "gztar", root_dir=RESULTS_DIR, base_dir=RUN_ID)
print("saved:", archive)
print("Commit config.json, env.json, summary.csv, analysis.json, crosscheck.json and the plot.")

## Stop when finished

The next cell shuts down vLLM. Then release the GPU itself:
- **Kaggle:** Run -> Stop Session.
- **Colab:** Runtime -> Disconnect and delete runtime.

Skipping this keeps spending your Kaggle hours or Colab units.

In [ ]:
anton("server", "stop", "--config", SERVER)